# Model Tester

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["XLA_FLAGS"] = (
    "--xla_gpu_cuda_data_dir=/hpc/mp/apps/nvidia/hpc_sdk/23.7/Linux_x86_64/23.7/cuda"
)

## Setup

In [ ]:
import sys
import time
import logging

import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import (
    EarlyStopping,
    TerminateOnNaN,
    TensorBoard,
    ModelCheckpoint,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import ExponentialDecay, LearningRateSchedule
from tensorflow.keras.metrics import RootMeanSquaredError

sys.path.append(os.path.join(os.getcwd(), ".."))
from scripts.models import AutoModel
from scripts.models.modelcore import get_model_class
from scripts.utils import (
    setup_logging,
    load_data,
    get_fisher,
    plot_predictions,
    plot_histogram,
)
from scripts.utils.tf import plot_metrics, TimedLoggingCallback, WarmupLearningRate

# tf.debugging.set_log_device_placement(True)

In [ ]:
print(f"TensorFlow version: {tf.__version__}")
print(f"CUDA version: {tf.sysconfig.get_build_info()['cuda_version']}")
print(f"cuDNN version: {tf.sysconfig.get_build_info()['cudnn_version']}")

In [ ]:
logger = setup_logging(__name__, base_level=logging.DEBUG)

## Configure

In [ ]:
args = [
    "settings/l500_n128.json",
    "--nsims",
    "100",
    "--narray",
    "10",
    # "--polarizations",
    # "TE",
]

MAX_EPOCHS = 30
BATCH_SIZE = 32

data_settings = {
    "shuffle": True,
    "shuffle_buffer": 1000,
    "seed": 0,
    "batch_size": BATCH_SIZE,
    "cache": True,
    "normalize": True,
    "channels_last": True,
}

callbacks = [
    EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True),
    # TensorBoard(log_dir=f"data/tensorboard/notebooks/{time.strftime('%Y%m%d-%H%M%S')}"),
    # TerminateOnNaN(),
]

In [ ]:
# from scripts.utils.tf import try_init_wandb
# try_init_wandb(notes="model testing", tags=["notebook"], append_to=callbacks)

## Isensee Model

In [ ]:
model = AutoModel(args + ["--model", "ISENSEE_V2"])

In [ ]:
train_ds, test_ds, val_ds = model.init_dataset(**data_settings).get_split(0.8, 0.1, 0.1)

In [ ]:
learning_rate = ExponentialDecay(1e-3, 10000, 0.96)
# learning_rate = WarmupLearningRate(warmup_steps=1000)

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    metrics = [RootMeanSquaredError()]
    opt = Adam(learning_rate)
    model.make_model()
    model.compile(optimizer=opt, loss="mse", metrics=metrics)
    model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
fisher = get_fisher(model.alm_file)

model.evaluate(test_ds, verbose=1)
y_pred = model.predict(test_ds, verbose=1).flatten()
y_test = np.concatenate([y.numpy() for _, y in test_ds])

plot_metrics(history, metrics=["loss"])
plot_predictions(y_test, y_pred, fisher=fisher)
plot_histogram(y_test, y_pred)

## ALM Model

In [ ]:
model = get_model_class("ALM")(args)
train_ds, test_ds, val_ds = model.init_dataset(**data_settings).get_split(0.8, 0.1, 0.1)

learning_rate = ExponentialDecay(1e-3, 10000, 0.96)

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    metrics = [RootMeanSquaredError()]
    opt = Adam(learning_rate)
    model.make_model()
    model.compile(optimizer=opt, loss="mse", metrics=metrics)
    model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
fisher = get_fisher(model.alm_file)

model.evaluate(test_ds, verbose=1)
y_pred = model.predict(test_ds, verbose=1).flatten()
y_test = np.concatenate([y.numpy() for _, y in test_ds]).flatten()

plot_metrics(history, metrics=["loss"])
plot_predictions(y_test, y_pred, fisher=fisher)
plot_histogram(y_test, y_pred)

## NAGARAJAPPA

In [ ]:
model = get_model_class("NAGARAJAPPA")(args)
train_ds, test_ds, val_ds = model.init_dataset(**data_settings).get_split(0.8, 0.1, 0.1)

learning_rate = ExponentialDecay(1e-3, 10000, 0.96)

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    metrics = [RootMeanSquaredError()]
    opt = Adam(learning_rate)
    model.make_model()
    model.compile(optimizer=opt, loss="mse", metrics=metrics)
    model.summary()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=callbacks,
    verbose=0,
)

In [ ]:
fisher = get_fisher(model.alm_file)

model.evaluate(test_ds, verbose=1)
y_pred = model.predict(test_ds, verbose=1).flatten()
y_test = np.concatenate([y.numpy() for _, y in test_ds])

plot_metrics(history, metrics=["loss"])
plot_predictions(y_test, y_pred, fisher=fisher)
plot_histogram(y_test, y_pred)